In [ ]:
import os, glob, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RS = 42
np.random.seed(RS)
tf.keras.utils.set_random_seed(RS)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!wget -q https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/PRCP-1002-HandwrittenDigits.zip -O digits.zip
!unzip -o -q digits.zip -d mnist_data
!find mnist_data -type f | head -30

In [ ]:
train_path, test_path = None, None
for p in glob.glob('mnist_data/**/*.csv', recursive=True):
    cols = pd.read_csv(p, nrows=1).columns
    if 'label' in cols:
        train_path = p
    else:
        test_path = p

if train_path:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path) if test_path else None
    y_all = train_df['label'].values
    X_all = train_df.drop(columns='label').values.astype('float32')
    print('Loaded CSV data:', train_df.shape, '| unlabeled test:', None if test_df is None else test_df.shape)
else:
    # Fallback: built-in MNIST (70,000 labeled images)
    (xa, ya), (xb, yb) = keras.datasets.mnist.load_data()
    X_all = np.concatenate([xa, xb]).reshape(-1, 784).astype('float32')
    y_all = np.concatenate([ya, yb])
    train_df, test_df = None, None
    print('CSV not found, using keras MNIST:', X_all.shape)

In [ ]:
print('Images:', X_all.shape[0], '| Pixels per image:', X_all.shape[1])
print('Pixel range:', X_all.min(), 'to', X_all.max())
print('Missing values:', np.isnan(X_all).sum())
print('Duplicate images:', pd.DataFrame(X_all).duplicated().sum())
print('Share of zero pixels: {:.1%}'.format((X_all == 0).mean()))

In [ ]:
# Class distribution
counts = pd.Series(y_all).value_counts().sort_index()
plt.figure(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values, color='steelblue')
plt.title('Images per digit'); plt.xlabel('Digit'); plt.ylabel('Count')
plt.show()
print((counts / counts.sum()).round(3))

In [ ]:
# Sample images
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for d in range(10):
    idx = np.where(y_all == d)[0][:2]
    for r, i in enumerate(idx):
        axes[r, d].imshow(X_all[i].reshape(28, 28), cmap='gray')
        axes[r, d].axis('off')
        if r == 0: axes[r, d].set_title(f'Digit {d}')
plt.show()

In [ ]:
# Average image of each digit and pixel importance
fig, axes = plt.subplots(1, 10, figsize=(15, 2.5))
for d in range(10):
    axes[d].imshow(X_all[y_all == d].mean(axis=0).reshape(28, 28), cmap='magma')
    axes[d].set_title(str(d)); axes[d].axis('off')
plt.suptitle('Mean image per digit'); plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(X_all.std(axis=0).reshape(28, 28), cmap='viridis'); plt.colorbar()
plt.title('Pixel standard deviation (which pixels carry information)'); plt.axis('off')
plt.show()

In [ ]:
X_all = X_all / 255.0          # scale pixels to 0-1

X_trval, X_test, y_trval, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=RS)
X_train, X_val, y_train, y_val = train_test_split(
    X_trval, y_trval, test_size=0.1765, stratify=y_trval, random_state=RS)

print('Train:', X_train.shape, '| Validation:', X_val.shape, '| Test:', X_test.shape)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=300, n_jobs=-1),
    'KNN (k=3, PCA)': Pipeline([('pca', PCA(n_components=0.95, random_state=RS)),
                                ('clf', KNeighborsClassifier(n_neighbors=3, n_jobs=-1))]),
    'Decision Tree': DecisionTreeClassifier(random_state=RS),
    'Random Forest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RS),
    'SVM (RBF, PCA)': Pipeline([('pca', PCA(n_components=0.95, random_state=RS)),
                                ('clf', SVC(kernel='rbf', C=10, gamma='scale'))]),
    'MLP (sklearn)': MLPClassifier(hidden_layer_sizes=(256, 128), early_stopping=True,
                                   max_iter=50, random_state=RS),
}

rows, fitted = [], {}
for name, model in models.items():
    t0 = time.perf_counter(); model.fit(X_train, y_train); fit_t = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(X_val);  pred_t = time.perf_counter() - t0
    rows.append({
        'Model': name,
        'Train Acc (5k sample)': accuracy_score(y_train[:5000], model.predict(X_train[:5000])),
        'Val Acc': accuracy_score(y_val, pred),
        'Val Macro-F1': f1_score(y_val, pred, average='macro'),
        'Fit time (s)': fit_t,
        'Predict time (s)': pred_t,
    })
    fitted[name] = model
    print(f'done: {name} -> val acc {rows[-1]["Val Acc"]:.4f}')

results = pd.DataFrame(rows)

In [ ]:
Xtr_img = X_train.reshape(-1, 28, 28, 1)
Xva_img = X_val.reshape(-1, 28, 28, 1)
Xte_img = X_test.reshape(-1, 28, 28, 1)

def build_cnn():
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        # light augmentation (only active during training)
        layers.RandomRotation(0.05, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.08, 0.08, fill_mode='constant', fill_value=0.0),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(), layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(), layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax'),
    ])

cnn = build_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]

t0 = time.perf_counter()
history = cnn.fit(Xtr_img, y_train, validation_data=(Xva_img, y_val),
                  epochs=30, batch_size=128, callbacks=callbacks, verbose=2)
cnn_fit_t = time.perf_counter() - t0

In [ ]:
# Learning curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='train'); ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(history.history['loss'], label='train'); ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Loss'); ax[1].legend()
plt.show()

In [ ]:
cnn_val_pred = np.argmax(cnn.predict(Xva_img, verbose=0), axis=1)
t0 = time.perf_counter(); cnn.predict(Xva_img, verbose=0); cnn_pred_t = time.perf_counter() - t0
cnn_train_pred = np.argmax(cnn.predict(Xtr_img[:5000], verbose=0), axis=1)

results = pd.concat([results, pd.DataFrame([{
    'Model': 'CNN (Keras)',
    'Train Acc (5k sample)': accuracy_score(y_train[:5000], cnn_train_pred),
    'Val Acc': accuracy_score(y_val, cnn_val_pred),
    'Val Macro-F1': f1_score(y_val, cnn_val_pred, average='macro'),
    'Fit time (s)': cnn_fit_t,
    'Predict time (s)': cnn_pred_t,
}])], ignore_index=True)

results = results.sort_values('Val Acc', ascending=False).reset_index(drop=True).round(4)
results

In [ ]:
results.plot(x='Model', y='Val Acc', kind='barh', legend=False, figsize=(8, 4),
             xlim=(results['Val Acc'].min() - 0.01, 1.0), title='Validation accuracy by model')
plt.show()

In [ ]:
best_name = results.loc[0, 'Model']
print('Best model by validation accuracy:', best_name)

if best_name == 'CNN (Keras)':
    test_pred = np.argmax(cnn.predict(Xte_img, verbose=0), axis=1)
else:
    test_pred = fitted[best_name].predict(X_test)

print('Test accuracy :', round(accuracy_score(y_test, test_pred), 4))
print('Test macro-F1 :', round(f1_score(y_test, test_pred, average='macro'), 4))
print(classification_report(y_test, test_pred, digits=4))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion matrix: {best_name}')
plt.show()

In [ ]:
# What does the model get wrong?
wrong = np.where(test_pred != y_test)[0]
print(f'{len(wrong)} mistakes out of {len(y_test)}')
fig, axes = plt.subplots(2, 8, figsize=(15, 4.5))
for ax, i in zip(axes.ravel(), wrong[:16]):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'T:{y_test[i]} P:{test_pred[i]}', fontsize=9); ax.axis('off')
plt.show()

In [ ]:
if best_name == 'CNN (Keras)':
    cnn.save('digit_model.keras')
else:
    import joblib; joblib.dump(fitted[best_name], 'digit_model.joblib')

In [ ]:
# Predict the unlabeled test file, if it exists
if test_df is not None and best_name == 'CNN (Keras)':
    X_sub = (test_df.values.astype('float32') / 255.0).reshape(-1, 28, 28, 1)
    sub_pred = np.argmax(cnn.predict(X_sub, verbose=0), axis=1)
    submission = pd.DataFrame({'ImageId': np.arange(1, len(sub_pred) + 1), 'Label': sub_pred})
    submission.to_csv('predictions.csv', index=False)
    print(submission.head())

In [ ]:
# Predict from your own image file (white digit on black background works best)
from PIL import Image, ImageOps

def predict_digit(path, invert=False):
    img = Image.open(path).convert('L').resize((28, 28))
    if invert:                      # use True for black ink on a white page
        img = ImageOps.invert(img)
    arr = np.array(img, dtype='float32').reshape(1, 28, 28, 1) / 255.0
    probs = cnn.predict(arr, verbose=0)[0]
    return int(probs.argmax()), float(probs.max())

# digit, confidence = predict_digit('my_digit.png', invert=True)

Model Comparison Report (fill in your own numbers): “Seven classifiers were compared on a stratified 70/15/15 split using validation accuracy and macro-F1. Logistic Regression and the Decision Tree performed worst because they can’t capture the shape of strokes. Random Forest, KNN and the RBF SVM reached about [x]%. The CNN performed best with [x]% validation and [x]% test accuracy, with a small train-validation gap thanks to dropout and augmentation. I recommend the CNN for production because it has the highest accuracy and fast inference, and the weakest digit pairs are visually ambiguous. If GPU or deployment size is a constraint, the PCA + SVM is a strong lightweight alternative.”
